In [1]:
import json
import os
import re
import argparse
import random
from icecream import ic


input_json_file = '/home/pc2041/VIP_lab/labelstudio/mmwhale2/work_dirs/iter8_completed.json'


ouptut_json_file = '/home/pc2041/VIP_lab/labelstudio/mmwhale2/work_dirs/vip_iter8_completed.json'

In [2]:
# categories = [{"id": 1, "name": "whale", "supercategory": "animal"},
#               {"id": 2, "name": "baby_whale", "supercategory": "animal"}]  # Adjust as per your dataset
categories = [{"id": 1, "name": "whale"}]  # Adjust as per your dataset
# category_name_to_id = {'whale': 1, 'baby_whale': 2, 'keypoint_whale': 1, 'keypoint_babywhale': 2}
category_name_to_id = {'whale': 1, 'baby_whale': 1, 'keypoint_whale': 1, 'keypoint_babywhale': 1}


def reverse_replace_string(input_str, replacement_mappings):
        reversed_mappings = {v: k for k, v in replacement_mappings.items()}
        pattern = re.compile('|'.join(re.escape(key) for key in reversed_mappings.keys()))
        result = pattern.sub(lambda x: reversed_mappings[x.group()], input_str)
        return result

def convert_percentage_to_pixels(value, original_size):
    return int(value * original_size/100)

def convert_rectangles_to_bbox(x_percent,y_percent,w_percent,h_percent, original_width, original_height):
    x = convert_percentage_to_pixels(x_percent, original_width)
    y = convert_percentage_to_pixels(y_percent, original_height)
    w = convert_percentage_to_pixels(w_percent, original_width)
    h = convert_percentage_to_pixels(h_percent, original_width)

    return x,y,w,h

def convert_keypoints_to_bbox(x_percent,y_percent,bbox_width,bbox_height, original_width, original_height):
    x = convert_percentage_to_pixels(x_percent, original_width)
    y = convert_percentage_to_pixels(y_percent, original_height)
    w = bbox_width
    h = bbox_height

    return x-w/2, y-h/2, w, h

def get_annotation_item(annotation, id, image_height, image_width, annotation_id, bbox_height, bbox_width):
    if annotation['type']== 'keypointlabels':
        category_name = annotation['value']['keypointlabels'][0]
        x,y,w,h = convert_keypoints_to_bbox(annotation['value']['x'],annotation['value']['y'],bbox_width,bbox_height, image_width, image_height)
    if annotation['type']== 'rectanglelabels':
        category_name = annotation['value']['rectanglelabels'][0]
        x,y,w,h = convert_rectangles_to_bbox(annotation['value']['x'],annotation['value']['y'],annotation['value']['width'], annotation['value']['height'], image_width, image_height)

    annotation_item = {
        "image_id": id,
        "category_id": category_name_to_id[category_name],
        "bbox": [
        x,
        y,
        w,
        h
        ],
        "iscrowd": 0,
        "area": h*w,
        "id": annotation_id}
    return annotation_item


In [3]:
with open(input_json_file, 'r') as file:
    tasks = json.load(file)

images = []
annotations = []

annotation_id = 0

replacement_mappings = {
    "Elements/High_Arctic_Cetacean_Survey_25mm/Plane1": "HACS2023DSLR_Plane1_yellow",
    "5C20230811_cam1": "5C20230811_cam1_25mm_nofoolography",
    "5C20230812_cam1": "5C20230812_cam1_25mm_nofoolography",
    "5C20230812_cam2": "5C20230812_cam2_25mm_foolography",
    "5C20230813_cam2": "5C20230813_cam2_25mm_foolography",
    "5C20230813_cam1": "5C20230813_cam1_25mm_nofoolography",
    "5C20230815_cam1": "5C20230815_cam1_25mm",
    "5C20230818_cam1": "5C20230818_cam1_25mm",
    "5C20230820_cam1": "5C20230820_cam1_25mm",
    "5C20230823_cam1": "5C20230823_cam1_25mm",
    "5C20230824_cam1": "5C20230824_cam1_25mm",
    "5C20230825_cam1": "5C20230825_cam1_25mm",
    "5C20230828_cam1": "5C20230828_cam1_25mm",
    "5C20230829_cam1": "5C20230829_cam1_25mm",
}


In [4]:
for task in tasks:
    # filename = task['data']['image'].replace("/data/local-files/?d=", "")
    # filename = os.path.join(image_filename_prefix, filename)
    filename = task['data']['image']
    filename = reverse_replace_string(filename, replacement_mappings)
    filename= filename.replace("%5C","/")
    task['data']['image'] = filename



In [5]:
with open(ouptut_json_file, 'w') as train_file:
    json.dump(tasks, train_file, indent=2)

ouptut_json_file

'/home/pc2041/VIP_lab/labelstudio/mmwhale2/work_dirs/vip_iter8_completed.json'